# 08 — Heatmap Visualization

Visualizes ML predictions as a heatmap overlaid on the Manhattan grid.

**Input:** `csv/07_predictions.csv`

**Output:**
- `outputs/latest/08_heatmap_predictions.png` — static matplotlib heatmap
- `outputs/latest/08_heatmap_interactive.html` — interactive folium Leaflet.js map

In [1]:
# ── Papermill parameters ──────────────────────────────
PLOTS_DIR = "outputs/latest"

In [2]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import json
import os

os.makedirs(PLOTS_DIR, exist_ok=True)

with open("grid.json", encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
INCLUDE_OTHER = config.get("include_other", False)
CSV_DIR = config.get("csv_dir", "csv")

# Load predictions
df = pd.read_csv(f"{CSV_DIR}/07_predictions.csv", dtype={"cell_id": str})
print(f"Loaded {len(df)} cell predictions")
print(f"Mode: {'3-class' if INCLUDE_OTHER else 'Binary'}")
for cls in df["predicted_zone"].unique():
    print(f"  {cls}: {(df['predicted_zone'] == cls).sum()}")

Loaded 47351 cell predictions
Mode: Binary
  Residential: 35707
  Commercial: 11644


In [3]:
# ── Compute cell rectangle dimensions in degrees ─────
import math

REF_LAT = df["cell_lat"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
HALF_LAT = LAT_STEP / 2
HALF_LON = LON_STEP / 2

print(f"Cell size: {LAT_STEP:.6f} lat x {LON_STEP:.6f} lon")
print(f"Lat range: {df['cell_lat'].min():.4f} - {df['cell_lat'].max():.4f}")
print(f"Lon range: {df['cell_lon'].min():.4f} - {df['cell_lon'].max():.4f}")

Cell size: 0.001351 lat x 0.001559 lon
Lat range: 29.4998 - 30.1662
Lon range: -95.9436 - -94.9173


In [4]:
# ── Static matplotlib heatmap with city basemap ──────
import contextily as ctx
from matplotlib.colors import LinearSegmentedColormap, to_rgba

fig, ax = plt.subplots(figsize=(8, 18))

if INCLUDE_OTHER:
    # 3-class: categorical colors with confidence-based alpha
    CLASS_COLORS = {"Commercial": "#B2182B", "Residential": "#2166AC", "Other": "#1B7837"}
    for _, row in df.iterrows():
        base_color = CLASS_COLORS.get(row["predicted_zone"], "#999999")
        conf = row.get("confidence", 0.7)
        alpha = 0.4 + 0.5 * conf
        rect = mpatches.Rectangle(
            (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
            LON_STEP, LAT_STEP, linewidth=0.1, edgecolor="gray",
            facecolor=base_color, alpha=alpha)
        ax.add_patch(rect)
else:
    # Binary: diverging colormap blue → red
    cmap = LinearSegmentedColormap.from_list("res_com", ["#2166AC", "#F7F7F7", "#B2182B"])
    for _, row in df.iterrows():
        prob_com = row.get("prob_commercial", 0.5)
        rect = mpatches.Rectangle(
            (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
            LON_STEP, LAT_STEP, linewidth=0.1, edgecolor="gray",
            facecolor=cmap(prob_com), alpha=0.75)
        ax.add_patch(rect)

# Set bounds with padding
pad = 0.005
ax.set_xlim(df["cell_lon"].min() - pad, df["cell_lon"].max() + pad)
ax.set_ylim(df["cell_lat"].min() - pad, df["cell_lat"].max() + pad)
ax.set_aspect("equal")

# Add subtle city basemap
try:
    ctx.add_basemap(ax, crs="EPSG:4326",
                    source=ctx.providers.CartoDB.PositronNoLabels,
                    zoom=14, alpha=0.4)
except Exception as e:
    print(f"Basemap download failed (needs internet): {e}")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Title with percentages
counts = df["predicted_zone"].value_counts()
class_list = ["Commercial", "Residential", "Other"] if INCLUDE_OTHER else ["Commercial", "Residential"]
pct_parts = " \u00b7 ".join(
    f"{cls} {100 * counts.get(cls, 0) / len(df):.1f}%"
    for cls in class_list if counts.get(cls, 0) > 0)
mode_label = "3-class" if INCLUDE_OTHER else "Binary"
ax.set_title(
    f"Manhattan Grid \u2014 Zone Predictions ({mode_label})\n"
    f"{len(df)} cells ({CELL_SIZE_M}m) \u00b7 {pct_parts}", fontsize=12)

# Legend / Colorbar
if INCLUDE_OTHER:
    legend_patches = [mpatches.Patch(color=c, label=l) for l, c in CLASS_COLORS.items()
                      if l in counts.index]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=10,
              framealpha=0.9, edgecolor="gray")
else:
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.3, pad=0.02)
    cbar.set_label("P(Commercial)", fontsize=10)

n_com_pred = counts.get("Commercial", 0)
n_res_pred = counts.get("Residential", 0)
pct_com = 100 * n_com_pred / len(df)
pct_res = 100 * n_res_pred / len(df)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_heatmap_predictions.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/08_heatmap_predictions.png")

Basemap download failed (needs internet): HTTPSConnectionPool(host='a.basemaps.cartocdn.com', port=443): Max retries exceeded with url: /light_nolabels/14/3826/6766.png (Caused by ConnectTimeoutError(<HTTPSConnection(host='a.basemaps.cartocdn.com', port=443) at 0x13c09dba0d0>, 'Connection to a.basemaps.cartocdn.com timed out. (connect timeout=None)'))
Saved: outputs/latest/08_heatmap_predictions.png


C:\Users\User\AppData\Local\Temp\ipykernel_23340\2281632101.py:77: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── Interactive folium map ────────────────────────────
try:
    import folium
    from folium import Rectangle

    m = folium.Map(
        location=[REF_LAT, df["cell_lon"].mean()],
        zoom_start=13,
        tiles="CartoDB positron",
    )

    if INCLUDE_OTHER:
        _COLORS = {"Commercial": "#B2182B", "Residential": "#2166AC", "Other": "#1B7837"}
    else:
        _COLORS = None  # use prob-based coloring

    for _, row in df.iterrows():
        bounds = [
            [row["cell_lat"] - HALF_LAT, row["cell_lon"] - HALF_LON],
            [row["cell_lat"] + HALF_LAT, row["cell_lon"] + HALF_LON],
        ]
        prob_com = row.get("prob_commercial", 0.5)

        if INCLUDE_OTHER:
            color = _COLORS.get(row["predicted_zone"], "#999999")
            conf = row.get("confidence", 0.7)
            opacity = 0.5 + 0.4 * conf
            popup_text = (f"<b>{row['cell_id']}</b><br>"
                          f"Actual: {row['zone_type']}<br>"
                          f"Predicted: <b>{row['predicted_zone']}</b><br>"
                          f"Confidence: {conf:.0%}")
        else:
            color = "#B2182B" if prob_com >= 0.5 else "#2166AC"
            opacity = 0.4 + 0.5 * abs(prob_com - 0.5) * 2
            popup_text = (f"<b>{row['cell_id']}</b><br>"
                          f"Actual: {row['zone_type']}<br>"
                          f"Predicted: <b>{row['predicted_zone']}</b><br>"
                          f"P(Commercial): {prob_com:.0%}")

        Rectangle(
            bounds=bounds,
            color="gray", weight=0.3,
            fill=True, fill_color=color, fill_opacity=opacity,
            popup=folium.Popup(popup_text, max_width=200),
        ).add_to(m)

    # Legend
    if INCLUDE_OTHER:
        legend_html = """
        <div style="position:fixed; bottom:30px; left:10px; z-index:1000;
                    background:white; padding:10px; border-radius:5px;
                    border:1px solid gray; font-size:13px;">
        <b>Zone Type</b><br>
        <span style="color:#B2182B;">&#9632;</span> Commercial<br>
        <span style="color:#2166AC;">&#9632;</span> Residential<br>
        <span style="color:#1B7837;">&#9632;</span> Other
        </div>
        """
    else:
        legend_html = """
        <div style="position:fixed; bottom:30px; left:10px; z-index:1000;
                    background:white; padding:10px; border-radius:5px;
                    border:1px solid gray; font-size:13px;">
        <b>Zone Type</b><br>
        <span style="color:#B2182B;">&#9632;</span> Commercial<br>
        <span style="color:#2166AC;">&#9632;</span> Residential
        </div>
        """
    m.get_root().html.add_child(folium.Element(legend_html))

    html_path = f"{PLOTS_DIR}/08_heatmap_interactive.html"
    m.save(html_path)
    print(f"Saved: {html_path}")
    print("Open in browser for interactive exploration.")

except ImportError:
    print("folium not installed — skipping interactive map.")

Saved: outputs/latest/08_heatmap_interactive.html
Open in browser for interactive exploration.


In [6]:
# ── Summary dashboard plot ────────────────────────────

_map = {"Commercial": "Commercial", "Mixed-Use": "Residential", "Residential": "Residential"}
if INCLUDE_OTHER:
    _map.update({"Institutional": "Other", "Open Space": "Other",
                 "Industrial": "Other", "Infrastructure": "Other"})
df["actual_class"] = df["zone_type"].map(_map)

CLASS_ORDER = ["Commercial", "Residential", "Other"] if INCLUDE_OTHER else ["Commercial", "Residential"]
CLASS_COLS = {"Commercial": "#B2182B", "Residential": "#2166AC", "Other": "#1B7837"}

n_total = len(df)
counts_pred = df["predicted_zone"].value_counts()
counts_actual = df["actual_class"].value_counts()
accuracy = (df["predicted_zone"] == df["actual_class"]).mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
mode_label = "3-class" if INCLUDE_OTHER else "Binary"
fig.suptitle(
    f"Grid Finding \u2014 Summary Dashboard ({mode_label})\n"
    f"{n_total} cells \u00b7 {CELL_SIZE_M}m grid \u00b7 Overall accuracy: {accuracy:.1%}",
    fontsize=14, fontweight="bold")

# (0,0) Predicted class distribution — pie chart
ax = axes[0, 0]
sizes = [counts_pred.get(c, 0) for c in CLASS_ORDER]
labels = [f"{c}\n{counts_pred.get(c, 0)} ({100*counts_pred.get(c, 0)/n_total:.1f}%)"
          for c in CLASS_ORDER]
colors = [CLASS_COLS[c] for c in CLASS_ORDER]
ax.pie(sizes, labels=labels, colors=colors, autopct="", startangle=90,
       textprops={"fontsize": 10}, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Predicted Distribution", fontsize=12)

# (0,1) Actual vs Predicted — grouped bar chart
ax = axes[0, 1]
x = np.arange(len(CLASS_ORDER))
width = 0.35
bars_actual = [counts_actual.get(c, 0) for c in CLASS_ORDER]
bars_pred = [counts_pred.get(c, 0) for c in CLASS_ORDER]
ax.bar(x - width/2, bars_actual, width, label="Actual (PLUTO)",
       color=[CLASS_COLS[c] for c in CLASS_ORDER], alpha=0.5)
ax.bar(x + width/2, bars_pred, width, label="Predicted (ML)",
       color=[CLASS_COLS[c] for c in CLASS_ORDER], alpha=1.0)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_ORDER)
ax.set_ylabel("Cell count")
ax.set_title("Actual vs Predicted", fontsize=12)
ax.legend()
for i, (a, p) in enumerate(zip(bars_actual, bars_pred)):
    ax.text(i - width/2, a + 10, str(a), ha="center", fontsize=9)
    ax.text(i + width/2, p + 10, str(p), ha="center", fontsize=9)

# (1,0) Probability / confidence histogram
ax = axes[1, 0]
if INCLUDE_OTHER and "confidence" in df.columns:
    for cls in CLASS_ORDER:
        subset = df[df["predicted_zone"] == cls]["confidence"]
        if len(subset) > 0:
            ax.hist(subset, bins=20, color=CLASS_COLS[cls], alpha=0.6, label=cls)
    ax.set_xlabel("Confidence (max probability)")
    ax.set_ylabel("Cell count")
    ax.set_title("Confidence Distribution by Class", fontsize=12)
    ax.legend()
elif "prob_commercial" in df.columns:
    ax.hist(df["prob_commercial"], bins=30, color="#666666", alpha=0.7, edgecolor="white")
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Decision boundary")
    ax.set_xlabel("P(Commercial)")
    ax.set_ylabel("Cell count")
    ax.set_title("Probability Distribution", fontsize=12)
    ax.legend()

# (1,1) Key feature means by predicted class
ax = axes[1, 1]
key_feats = ["amenity_density", "shop_density_km2", "tourism_density",
             "brand_ratio", "landuse_entropy"]
available_feats = [f for f in key_feats if f in df.columns]
if available_feats:
    means = df.groupby("predicted_zone")[available_feats].mean()
    y_pos = np.arange(len(available_feats))
    bar_h = 0.8 / len(CLASS_ORDER)
    for i, cls in enumerate(CLASS_ORDER):
        if cls in means.index:
            ax.barh(y_pos + i * bar_h, means.loc[cls, available_feats],
                    bar_h, label=cls, color=CLASS_COLS[cls], alpha=0.85)
    ax.set_yticks(y_pos + bar_h * len(CLASS_ORDER) / 2)
    ax.set_yticklabels(available_feats, fontsize=9)
    ax.set_xlabel("Mean value")
    ax.set_title("Feature Means by Predicted Class", fontsize=12)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/09_summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/09_summary_dashboard.png")

Saved: outputs/latest/09_summary_dashboard.png


C:\Users\User\AppData\Local\Temp\ipykernel_23340\3825176709.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Summary statistics ────────────────────────────────
print("Prediction summary:")
print(f"  Mode: {'3-class' if INCLUDE_OTHER else 'Binary'}")
print(f"  Total cells: {len(df)}")
print(f"  Predicted Commercial: {n_com_pred} ({pct_com:.1f}%)")
print(f"  Predicted Residential: {n_res_pred} ({pct_res:.1f}%)")
if INCLUDE_OTHER:
    n_oth = counts.get("Other", 0)
    print(f"  Predicted Other: {n_oth} ({100*n_oth/len(df):.1f}%)")
print(f"  Mean P(Commercial): {df['prob_commercial'].mean():.3f}")
print(f"  Median P(Commercial): {df['prob_commercial'].median():.3f}")
print(f"\n  Overall accuracy (on all cells): {accuracy:.3f}")

Prediction summary:
  Mode: Binary
  Total cells: 47351
  Predicted Commercial: 11644 (24.6%)
  Predicted Residential: 35707 (75.4%)
  Mean P(Commercial): 0.298
  Median P(Commercial): 0.143

  Overall accuracy (on all cells): 0.778
